# P1-E16.7H-1 Renamed (P1-E11.H_1) FINAL CORRECTED — Exact P1-E07 Automated-QA Association and Threshold-Sensitivity Audit

This analysis uses the exact full-cohort P1-E07 artifacts: the 979-recording labelled audit, candidate-epoch quantitative QA measurements, and 914-recording retained manifest. It tests diagnosis–retention association and prespecified threshold perturbations. QA was automated, and labels were joined only after eligibility was frozen; no blinded specialist QA was performed.

## 1. Configuration and fixed inputs

In [ ]:
from pathlib import Path
from datetime import datetime,timezone
import hashlib,json,sys,platform
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import chi2_contingency
from sklearn.metrics import balanced_accuracy_score,f1_score,matthews_corrcoef

SEED=20261012;BOOTSTRAP_REPS=5000
PAPER_ROOT=Path(r'D:\EEG_AD_Research\paper1_experiments\paper1_outputs')
CURRENT_ROOT=Path(r'D:\EEG_AD_Research\dementia\paper one\paper1_outputs')
E06_ROOT=PAPER_ROOT/'p1_e06_1_automated_qa_preprocessing_pilot'
E07_ROOT=PAPER_ROOT/'p1_e07_1_full_cohort_reference_preprocessing_automated_qa'
E16_7BR_ROOT=CURRENT_ROOT/'p1_e16_7b_r_unified_four_endpoint_nested_method_comparison'
OUTPUT_ROOT=CURRENT_ROOT/'p1_e16_7h_1_exact_p1e07_automated_qa_sensitivity'
TABLES=OUTPUT_ROOT/'tables';FIGURES=OUTPUT_ROOT/'figures'
for p in (OUTPUT_ROOT,TABLES,FIGURES):p.mkdir(parents=True,exist_ok=True)

E06_SPEC=E06_ROOT/'tables'/'10_LOCKED_AUTOMATED_PREPROCESSING_SPECIFICATION.csv'
RECORDING_AUDIT=E07_ROOT/'tables'/'06_recording_preprocessing_audit_with_labels.csv'
EPOCH_AUDIT=E07_ROOT/'tables'/'03_candidate_epoch_automatic_qa_audit.csv'
RETAINED_MANIFEST=E07_ROOT/'tables'/'09_LOCKED_PREPROCESSING_ELIGIBLE_RECORDING_MANIFEST.csv'
E07_INTEGRITY=E07_ROOT/'tables'/'11_final_integrity_checks.csv'
OOF_PATH=E16_7BR_ROOT/'tables'/'08_ALL_METHOD_NESTED_OOF_PREDICTIONS.csv'

LABELS=['Normal','MCI','AD'];ENDPOINTS={'normal_vs_ad':['Normal','AD'],'mci_vs_ad':['MCI','AD'],'normal_vs_mci':['Normal','MCI'],'three_class':['Normal','MCI','AD']}
SCENARIOS={
 'locked':{'ptp':600.,'rms':120.,'extreme_fraction':.020,'robust':5.0,'minimum_epochs':6},
 'stricter_10pct':{'ptp':540.,'rms':108.,'extreme_fraction':.018,'robust':4.5,'minimum_epochs':7},
 'relaxed_10pct':{'ptp':660.,'rms':132.,'extreme_fraction':.022,'robust':5.5,'minimum_epochs':5}}

def sha256_file(path,chunk=2**20):
 h=hashlib.sha256()
 with open(path,'rb') as f:
  for b in iter(lambda:f.read(chunk),b''):h.update(b)
 return h.hexdigest()
def save_table(d,n):p=TABLES/f'{n}.csv';d.to_csv(p,index=False);return p
def normalize_id(x):
 s=str(x).strip();return s[:-2].zfill(5) if s.endswith('.0') and s[:-2].isdigit() else (s.zfill(5) if s.isdigit() else s)
def normalize_label(x):
 s=str(x).strip().lower().replace('_',' ').replace('-',' ');a={'normal':'Normal','control':'Normal','healthy':'Normal','hc':'Normal','mci':'MCI','mild cognitive impairment':'MCI','ad':'AD','alzheimer disease':'AD',"alzheimer's disease":'AD'};return a.get(s,str(x).strip())
def to_bool(s):return pd.Series(s).astype(str).str.strip().str.lower().isin(['true','1','yes','pass','passed','eligible','accepted']).to_numpy()
def savefig(fig,n):fig.tight_layout();fig.savefig(FIGURES/f'{n}.pdf',bbox_inches='tight');fig.savefig(FIGURES/f'{n}.png',dpi=600,bbox_inches='tight',facecolor='white')
sns.set_theme(style='whitegrid',context='paper',font_scale=1.1)
print('Output:',OUTPUT_ROOT)

## 2. Authenticate exact upstream artifacts

In [ ]:
required=[E06_SPEC,RECORDING_AUDIT,EPOCH_AUDIT,RETAINED_MANIFEST,E07_INTEGRITY,OOF_PATH]
missing=[str(p) for p in required if not p.is_file()]
if missing:raise FileNotFoundError('Missing exact required artifact(s):\n'+'\n'.join(missing))
inputs=pd.DataFrame([{'role':r,'path':str(p),'sha256':sha256_file(p),'size_bytes':p.stat().st_size} for r,p in zip(
 ['P1-E06 locked automated-QA specification','P1-E07 979-recording labelled audit','P1-E07 candidate-epoch QA audit','P1-E07 914-recording retained manifest','P1-E07 integrity checks','nested OOF predictions'],required)])
save_table(inputs,'01_AUTHENTICATED_INPUTS');display(inputs)
up=pd.read_csv(E07_INTEGRITY);pc=next(c for c in up.columns if c.lower() in ['pass','passed'])
if not to_bool(up[pc]).all():raise AssertionError('P1-E07 upstream integrity lock failed.')

## 3. Load and reconcile the 979-to-914 cohort

In [ ]:
recordings=pd.read_csv(RECORDING_AUDIT,dtype={'recording_id':str});epochs=pd.read_csv(EPOCH_AUDIT,dtype={'recording_id':str});retained=pd.read_csv(RETAINED_MANIFEST,dtype={'recording_id':str})
for d in (recordings,epochs,retained):d['recording_id']=d.recording_id.map(normalize_id)
recordings['experimental_label']=recordings.experimental_label.map(normalize_label)
recordings['preprocessing_eligible']=to_bool(recordings.preprocessing_eligible)
epochs['accepted']=to_bool(epochs.accepted);epochs['finite']=to_bool(epochs.finite)
retained_ids=set(retained.recording_id)
checks=pd.DataFrame([
 {'check':'recording audit has 979 unique IDs','pass':len(recordings)==979 and recordings.recording_id.nunique()==979,'detail':len(recordings)},
 {'check':'retained manifest has 914 unique IDs','pass':len(retained_ids)==914,'detail':len(retained_ids)},
 {'check':'retained IDs subset of audit','pass':retained_ids<=set(recordings.recording_id),'detail':len(retained_ids-set(recordings.recording_id))},
 {'check':'recorded eligibility equals retained membership','pass':np.array_equal(recordings.preprocessing_eligible.to_numpy(),recordings.recording_id.isin(retained_ids).to_numpy()),'detail':'exact rowwise comparison'},
 {'check':'all diagnoses represented','pass':set(recordings.experimental_label)==set(LABELS),'detail':recordings.experimental_label.value_counts().to_dict()},
 {'check':'epoch IDs covered by recording audit','pass':set(epochs.recording_id)<=set(recordings.recording_id),'detail':len(set(epochs.recording_id)-set(recordings.recording_id))},
])
save_table(checks,'02_COHORT_RECONCILIATION');display(checks)
if not checks['pass'].all():raise AssertionError('Exact P1-E07 cohort reconciliation failed.')

## 4. Reconstruct the locked automated-QA rule

The epoch decision requires finite values, zero flat channels, peak-to-peak below 600 µV, RMS below 120 µV, extreme-amplitude fraction below 0.02, and robust multimetric score below 5. A recording is retained when at least six epochs pass; processing failures remain excluded.

In [ ]:
def epoch_pass(d,t):
 return (d.finite & (pd.to_numeric(d.flat_channels,errors='coerce')==0) &
         (pd.to_numeric(d.ptp_uv,errors='coerce')<t['ptp']) &
         (pd.to_numeric(d.rms_uv,errors='coerce')<t['rms']) &
         (pd.to_numeric(d.extreme_fraction,errors='coerce')<t['extreme_fraction']) &
         (pd.to_numeric(d.robust_multimetric_score,errors='coerce')<t['robust']))

locked_recomputed=epoch_pass(epochs,SCENARIOS['locked'])
epoch_rule_match=np.array_equal(locked_recomputed.to_numpy(),epochs.accepted.to_numpy())
locked_counts=locked_recomputed.groupby(epochs.recording_id).sum()
processing_success=to_bool(recordings.processing_success)
recordings['recomputed_locked']=(recordings.recording_id.map(locked_counts).fillna(0).to_numpy()>=6)&processing_success
reconstruction=pd.DataFrame([
 {'check':'epoch-level locked decisions reproduced','pass':epoch_rule_match,'detail':int((locked_recomputed.to_numpy()!=epochs.accepted.to_numpy()).sum())},
 {'check':'recording-level 914 retained reproduced','pass':int(recordings.recomputed_locked.sum())==914,'detail':int(recordings.recomputed_locked.sum())},
 {'check':'recording decisions match P1-E07','pass':np.array_equal(recordings.recomputed_locked,recordings.preprocessing_eligible),'detail':int((recordings.recomputed_locked!=recordings.preprocessing_eligible).sum())}])
save_table(reconstruction,'03_LOCKED_QA_RULE_RECONSTRUCTION');display(reconstruction)
if not reconstruction['pass'].all():raise AssertionError('Locked automated-QA rule could not be reproduced exactly.')

## 5. Diagnosis–retention association

In [ ]:
cont=pd.crosstab(recordings.experimental_label,recordings.preprocessing_eligible).reindex(index=LABELS,columns=[False,True],fill_value=0)
chi2,p,dof,expected=chi2_contingency(cont);n=cont.to_numpy().sum();v=np.sqrt(chi2/(n*min(cont.shape[0]-1,cont.shape[1]-1)))
omnibus=pd.DataFrame([{'test':'Pearson chi-square','n':n,'chi2':chi2,'df':dof,'p_value':p,'cramers_v':v}])
retention=(recordings.groupby('experimental_label').preprocessing_eligible.agg(['count','sum','mean']).reindex(LABELS).reset_index().rename(columns={'experimental_label':'diagnosis','count':'montage_eligible_n','sum':'retained_n','mean':'retention_proportion'}));retention['rejected_n']=retention.montage_eligible_n-retention.retained_n

rng=np.random.default_rng(SEED);rows=[]
for i,a in enumerate(LABELS):
 for b in LABELS[i+1:]:
  xa=recordings.loc[recordings.experimental_label==a,'preprocessing_eligible'].to_numpy(float);xb=recordings.loc[recordings.experimental_label==b,'preprocessing_eligible'].to_numpy(float)
  boot=np.asarray([rng.choice(xa,len(xa),True).mean()-rng.choice(xb,len(xb),True).mean() for _ in range(BOOTSTRAP_REPS)])
  pv=max(1/BOOTSTRAP_REPS,2*min((boot<=0).mean(),(boot>=0).mean()))
  rows.append({'diagnosis_a':a,'diagnosis_b':b,'retention_a':xa.mean(),'retention_b':xb.mean(),'risk_difference':xa.mean()-xb.mean(),'ci95_low':np.quantile(boot,.025),'ci95_high':np.quantile(boot,.975),'p_value':pv})
pairwise=pd.DataFrame(rows).sort_values('p_value').reset_index(drop=True);m=len(pairwise);pairwise['holm_adjusted_p_value']=np.maximum.accumulate(np.minimum(1,(m-np.arange(m))*pairwise.p_value));pairwise['holm_significant']=pairwise.holm_adjusted_p_value<.05
save_table(retention,'04_RETENTION_BY_DIAGNOSIS');save_table(omnibus,'05_OMNIBUS_DIAGNOSIS_RETENTION_TEST');save_table(pairwise,'06_PAIRWISE_RETENTION_DIFFERENCES');display(retention);display(omnibus);display(pairwise)

## 6. Exclusion mechanisms by diagnosis

In [ ]:
reasons=(recordings.groupby(['experimental_label','reason']).size().rename('n').reset_index().rename(columns={'experimental_label':'diagnosis'}));den=recordings.groupby('experimental_label').size().rename('eligible_n');reasons=reasons.join(den,on='diagnosis');reasons['proportion_of_eligible']=reasons.n/reasons.eligible_n
save_table(reasons,'07_EXCLUSION_REASONS_BY_DIAGNOSIS');display(reasons)
fig,ax=plt.subplots(figsize=(8,4.5));plot=retention.assign(retention_percent=lambda x:100*x.retention_proportion);sns.barplot(data=plot,x='diagnosis',y='retention_percent',hue='diagnosis',legend=False,ax=ax);ax.set_ylim(0,100);ax.set_ylabel('Retained recordings (%)');ax.set_xlabel('Diagnosis');ax.set_title('Automated-QA retention by diagnosis',fontweight='bold');[ax.text(p.get_x()+p.get_width()/2,p.get_height()+1,f'{p.get_height():.1f}%',ha='center') for p in ax.patches];savefig(fig,'Figure_01_Automated_QA_Retention_by_Diagnosis');plt.show()

## 7. Threshold-sensitivity scenarios

The extreme-amplitude definition remains fixed at 150 µV because the stored `extreme_fraction` was calculated at that amplitude. Sensitivity scenarios perturb the allowable fraction and other stored thresholds, plus the minimum accepted-epoch count. Processing failures cannot be recovered by threshold relaxation.

In [ ]:
scenario_recordings=[];scenario_summary=[]
for name,t in SCENARIOS.items():
 passed=epoch_pass(epochs,t);counts=passed.groupby(epochs.recording_id).sum()
 keep=(recordings.recording_id.map(counts).fillna(0).to_numpy()>=t['minimum_epochs'])&processing_success
 recordings[name+'_retained']=keep
 for diagnosis in LABELS:
  q=recordings.experimental_label.eq(diagnosis);scenario_summary.append({'scenario':name,'diagnosis':diagnosis,'eligible_n':int(q.sum()),'retained_n':int(keep[q].sum()),'retention_proportion':float(keep[q].mean())})
 for rid,flag in zip(recordings.recording_id,keep):scenario_recordings.append({'scenario':name,'recording_id':rid,'retained':bool(flag),'accepted_epochs_under_scenario':int(counts.get(rid,0))})
scenario_summary=pd.DataFrame(scenario_summary);scenario_recordings=pd.DataFrame(scenario_recordings)
save_table(scenario_summary,'08_THRESHOLD_SCENARIO_RETENTION');save_table(scenario_recordings,'09_THRESHOLD_SCENARIO_RECORDING_DECISIONS');display(scenario_summary)
assert int(scenario_summary.query("scenario=='locked'").retained_n.sum())==914

## 8. Conditional stability of existing nested OOF predictions

Predictions are not retrained. They are re-scored on the subset of originally modelled recordings retained by each threshold scenario. Relaxed thresholds may add recordings without existing features or predictions; coverage is reported explicitly and no performance is imputed.

In [ ]:
oof=pd.read_csv(OOF_PATH,dtype={'recording_id':str});oof.recording_id=oof.recording_id.map(normalize_id);oof['task']=oof.task.astype(str).str.lower();oof['true_label']=oof.true_label.map(normalize_label);oof['predicted_label']=oof.predicted_label.map(normalize_label)
rows=[]
for scenario in SCENARIOS:
 keep=set(recordings.loc[recordings[scenario+'_retained'],'recording_id'])
 for (task,method),q0 in oof.groupby(['task','method']):
  q=q0[q0.recording_id.isin(keep)];classes=ENDPOINTS[task]
  evaluable=len(q)>0 and set(q.true_label)==set(classes)
  row={'scenario':scenario,'endpoint':task,'method':method,'scenario_retained_recordings':len(keep),'prediction_n':len(q),'prediction_coverage':len(q)/max(1,len(keep)),'evaluable':evaluable}
  if evaluable:row.update({'balanced_accuracy':balanced_accuracy_score(q.true_label,q.predicted_label),'macro_f1':f1_score(q.true_label,q.predicted_label,average='macro'),'mcc':matthews_corrcoef(q.true_label,q.predicted_label)})
  rows.append(row)
stability=pd.DataFrame(rows);save_table(stability,'10_CONDITIONAL_OOF_STABILITY');display(stability.head(40))

## 9. Publication figures

In [ ]:
fig,ax=plt.subplots(figsize=(9,4.8));plot=scenario_summary.assign(retention_percent=lambda x:100*x.retention_proportion);sns.pointplot(data=plot,x='scenario',y='retention_percent',hue='diagnosis',markers='o',linestyles='-',ax=ax);ax.set_ylabel('Retained recordings (%)');ax.set_xlabel('QA scenario');ax.set_title('Retention sensitivity to automated-QA thresholds',fontweight='bold');savefig(fig,'Figure_02_QA_Threshold_Sensitivity');plt.show()
conv=stability[stability.method.astype(str).str.contains('conventional',case=False,na=False)&stability.evaluable]
if len(conv):
 fig,ax=plt.subplots(figsize=(10,4.8));sns.pointplot(data=conv,x='endpoint',y='balanced_accuracy',hue='scenario',markers='o',linestyles='-',ax=ax);ax.set_ylim(.25,1);ax.set_ylabel('Balanced accuracy');ax.set_xlabel('Endpoint');ax.set_title('Conditional OOF stability under QA scenarios',fontweight='bold');savefig(fig,'Figure_03_Conditional_OOF_Stability');plt.show()

## 10. Final integrity and interpretation lock

In [ ]:
final=pd.DataFrame([
 {'check':'exact P1-E07 inputs authenticated','pass':True,'detail':len(inputs)},
 {'check':'979-recording denominator','pass':len(recordings)==979,'detail':len(recordings)},
 {'check':'914-recording locked retention reproduced','pass':int(recordings.locked_retained.sum())==914 if 'locked_retained' in recordings else int(recordings.preprocessing_eligible.sum())==914,'detail':int(recordings.preprocessing_eligible.sum())},
 {'check':'epoch rule exactly reproduced','pass':reconstruction['pass'].all(),'detail':reconstruction.to_dict('records')},
 {'check':'three diagnostic classes present','pass':set(recordings.experimental_label)==set(LABELS),'detail':recordings.experimental_label.value_counts().to_dict()},
 {'check':'all threshold scenarios evaluated','pass':set(scenario_summary.scenario)==set(SCENARIOS),'detail':sorted(scenario_summary.scenario.unique())},
 {'check':'no model fitting or selection performed','pass':True,'detail':'conditional rescoring only'},
 {'check':'no external outcomes accessed','pass':True,'detail':'CAUEEG QA/development artifacts only'},
 {'check':'no blinded specialist QA claimed','pass':True,'detail':'automated quantitative QA; specialist validation remains a limitation'}])
save_table(final,'11_FINAL_INTEGRITY_CHECKS');display(final)
if not final['pass'].all():raise AssertionError('P1-E16.7H-1 final integrity failure.')
manifest=[]
for p in sorted(OUTPUT_ROOT.rglob('*')):
 if p.is_file() and p.name!='12_OUTPUT_MANIFEST.csv':manifest.append({'path':str(p),'sha256':sha256_file(p),'size_bytes':p.stat().st_size})
save_table(pd.DataFrame(manifest),'12_OUTPUT_MANIFEST')
decision={'experiment':'P1-E16.7H-1','status':'PASSED — EXACT P1-E07 AUTOMATED-QA ASSOCIATION AND SENSITIVITY LOCKED','completed_utc':datetime.now(timezone.utc).isoformat(),'qa_type':'automated quantitative QA','blinded_specialist_qa_performed':False,'model_retraining_performed':False,'external_data_accessed':False,'interpretation':'Diagnosis-independent rule construction does not guarantee diagnosis-independent effects; association is not causation.'}
(OUTPUT_ROOT/'P1_E16_7H_1_FINAL_DECISION.json').write_text(json.dumps(decision,indent=2),encoding='utf-8');print(decision['status'])

## Interpretation boundary

The association analysis evaluates whether retention proportions differ by diagnosis after labels are joined for audit. It does not establish that QA caused bias. Threshold scenarios test robustness to reasonable perturbations of the stored quantitative rule; they are not optimized alternative pipelines. No blinded specialist QA was conducted.